# 00 · Getting started: the `Signal` and the CommKit conventions

This first notebook introduces the few ideas every other notebook relies on:

1. **Data** is a `Signal` (samples plus what is known about them) or a plain
   NumPy/CuPy array.
2. **Algorithms are plain functions**, `f(data, *, parameters)`, that return
   new data and never modify their input.
3. **Descriptions** (a constellation, a pulse, an algorithm) are small frozen
   objects written inline in the call: `RRC(rolloff=0.1)`, `BPS()`.
4. **The device follows the data**: build the arrays on the GPU and every
   later step runs there.

The notebooks are numbered in teaching order. 00-05 cover signals and DSP
fundamentals, 06-08 the channel and how performance is measured, 09-16 the
receiver algorithms, 17-19 optical fiber systems, and 20-25 laser and
carrier-phase characterization.  `examples/README.md` lists them all.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

import commkit as ck
from commkit import RRC, Constellation, Signal
from commkit.backend import to_device

# Build the data on the GPU when there is one; every later step follows it.
DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()

## 1. A `Signal` by hand

A `Signal` holds three kinds of information next to its samples:

| kind | fields | meaning |
|---|---|---|
| **facts** | `sampling_rate`, `symbol_rate`, `center_frequency`, `domain` | true properties of the waveform; `sps` is derived from the two rates, and `domain` (`"DIG"`, `"RF"` or `"OPT"`) says whether there is a carrier |
| **description** | `constellation`, `pulse` | how the waveform was built; later stages use them as defaults |
| **ground truth** | `reference` (symbols and bits), `frame` | what was transmitted, for metrics and data-aided algorithms |

Construction does no hidden work: it stores what it is given.  The samples
are `(N,)` for one channel or `(C, N)` for C channels; time is always the
last axis.

In [ ]:
fs = 1e6  # sampling rate [Hz]
t = np.arange(1000) / fs
tone = np.exp(2j * np.pi * 50e3 * t).astype(np.complex64)  # 50 kHz complex tone

sig = Signal(samples=tone, sampling_rate=fs, symbol_rate=fs / 10)
print(sig)
print("\nsps =", sig.sps, "| duration =", sig.duration, "s")

## 2. Generating a modulated waveform

Most work starts with `generate`: it draws random bits with a seeded `rng`,
maps them on the constellation, upsamples to `sps` samples per symbol and
shapes them with the pulse.  The result carries everything it knows: the
constellation, the pulse and the transmitted symbols and bits in
`reference`.

Two conventions hold for every generated waveform:

* **storage precision** is `complex64`: the single-precision path is what a
  GPU runs fastest, and it is ample for communication signals;
* **power normalization**: unit energy per symbol, so the average sample
  power is `1/sps` (and 1 at one sample per symbol).

In [ ]:
tx = ck.generate(
    Constellation.qam(16),
    num_symbols=4096,
    symbol_rate=10e9,
    sps=4,
    pulse=RRC(rolloff=0.2),
    rng=1,
    device=DEVICE,
)
print(tx)

x = to_device(tx.samples, "cpu")
print("\ndtype:", x.dtype)
print(f"mean sample power = {np.mean(np.abs(x) ** 2):.4f}   (1/sps = {1 / tx.sps:.4f})")
print(
    "reference:",
    tx.reference.symbols.shape,
    "symbols,",
    tx.reference.bits.shape,
    "bits",
)

## 3. Looking at a signal

The plotting functions take a `Signal` (or an array plus the facts it
lacks) and handle the device transfer themselves.  Every plotting function
starts with `plot_`, and nothing in the numerical code imports them.

In [ ]:
ck.plotting.plot_time_domain(tx, num_symbols=20, show=True)
ck.plotting.plot_psd(tx, nperseg=1024, show=True)
ck.plotting.plot_ideal_constellation(tx.constellation, show=True)

## 4. Functions return new data

A transform accepts an array or a `Signal` and returns the same kind: array
in, array out; `Signal` in, a new `Signal` out with the same facts.  The
input is never modified.  Here AWGN is added at Es/N0 = 15 dB; the noise
level follows from the signal's `sps`, which the function reads from the
`Signal`.

In [ ]:
rx = ck.impairments.apply_awgn(tx, esn0_db=15, rng=2)
print(
    type(rx).__name__,
    "| input unchanged:",
    rx is not tx and rx.samples is not tx.samples,
)

# The same function on a bare array: the facts it needs are now arguments.
rx_array = ck.impairments.apply_awgn(tx.samples, esn0_db=15, sps=4, rng=2)
print(type(rx_array).__name__)

## 5. Facts versus choices

Facts come from the `Signal`; an argument that contradicts one raises instead
of silently winning.  Choices (the decision constellation, the matched
filter's pulse, a noise variance) default to the `Signal`'s description, and
an explicit argument overrides them.

In [ ]:
try:
    ck.impairments.apply_phase_noise(tx, linewidth=1e5, sampling_rate=1e9, rng=3)
except ValueError as err:
    print("ValueError:", err)

# The matched filter takes the pulse from the Signal (a choice; pass pulse= to override).
mf = ck.filtering.matched_filter(rx)
y = ck.multirate.decimate_to_symbol_rate(mf)
print(
    "after decimation: sps =",
    y.sps,
    "| reference still aligned:",
    y.reference.symbols.shape,
)
ck.plotting.plot_constellation(y, overlay_ideal=True, show=True)

## 6. Measuring

Metrics compare against the reference the `Signal` carries and return plain
host floats (one per channel for multi-channel input).

In [ ]:
print(f"EVM = {ck.metrics.evm(y):.2f} %")
print(f"SNR = {ck.metrics.snr(y):.2f} dB  (Es/N0 was 15 dB)")
print(f"BER = {ck.metrics.ber(y):.2e}")

## 7. Devices

There is no global backend switch: functions run where their input lives.
`sig.to("gpu")` / `sig.to("cpu")` return a copy on the other device, and
`to_device(array, "cpu")` brings any array to the host, e.g. for your own
Matplotlib plots.  The factories that have no input to follow (`generate`,
`generate_phase_noise`, frames) take `device=` instead.

A seed gives the same transmitted bits on both devices.  Noise is drawn on
the data's device, so its realization differs between CPU and GPU while its
statistics agree.

In [ ]:
on_host = rx.to("cpu")
print("rx on", rx.backend, "| copy on", on_host.backend)

xp = rx.xp  # numpy or cupy, whichever holds the samples
print("array module:", xp.__name__)

## 8. Copies, edits and files

`replace(...)` returns a new `Signal` with some fields changed; the arrays
are shared, not copied.  `clone()` makes an independent deep copy.
`save_npz` / `load_npz` store a `Signal` with its metadata and reference.

In [ ]:
louder = rx.replace(samples=2 * rx.samples)
print("replace shares facts:", louder.sampling_rate == rx.sampling_rate)

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "capture.npz"
    ck.save_npz(rx, path)
    back = ck.load_npz(path, device=DEVICE)
print(back)

## 9. Diagnostics

The library logs through the standard `logging` module, to the logger
`"commkit"`, and never configures it on import.  Without configuration,
Python prints only warnings and errors (to stderr) and drops the `INFO` and
`DEBUG` messages, which is why a notebook shows nothing by default.

`ck.set_log_level("INFO")` attaches a console handler and shows one line per
call with what the algorithms found: the value a metric or estimator
measured, the parameters an impairment applied, an equalizer's exit MSE, how
many codewords failed a decoder's check.  `"DEBUG"` adds what each call is
about to do.  An application that configures `logging` itself (for example
`logging.basicConfig(level=logging.INFO)`) sees the same messages.

**Next:** [01 · Constellations and bit mapping](01-constellations-and-bit-mapping.ipynb).

In [ ]:
ck.set_log_level("INFO")
rx2 = ck.impairments.apply_awgn(tx, esn0_db=15, rng=2)
y2 = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(rx2))
snr_db = ck.metrics.snr(y2)

ck.set_log_level("WARNING")  # warnings only from here on
snr_db = ck.metrics.snr(y2)  # silent